# Lesson 01 - 人工智能代理简介

欢迎来到**人工智能代理初学者**课程的第一课！

**人工智能代理**是一个使用大型语言模型（LLM）作为推理引擎的程序，能够在现实世界中采取*行动*——调用API、查询数据库或运行代码——以代表用户完成目标。

在本笔记本中，您将构建第一个代理：一个推荐度假目的地的**旅行代理**。在此过程中，您将学习如何：

1. 使用**Microsoft Agent Framework**连接到 Azure AI Foundry Agent 服务。
2. 为代理提供一个**工具**——一个可以调用的普通 Python 函数。
3. 运行代理并检查其响应。
4. 逐令牌地流式传输代理的响应。


## Setup

在运行此笔记本之前，请确保您已经：

1. **拥有一个 Azure AI Foundry 项目**，并部署了聊天模型（例如 `gpt-4o-mini`）。
2. **已通过 Azure CLI 登录** — 在终端中运行 `az login`。
3. **设置所需的环境变量：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 项目端点。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您已部署模型的名称。

下面的单元格将安装您需要的 Python 包。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 创建您的第一个智能体

智能体需要两样东西：

- **指令**，告诉它*它是谁*以及*如何行为*（系统提示）。
- **工具** — 用 `@tool` 装饰的 Python 函数，智能体可以调用这些函数来获取信息或执行操作。

下面我们定义了一个简单的工具，返回一个受欢迎的度假目的地列表。当用户请求旅行推荐时，智能体将使用此工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 流式响应

为了获得更具互动性的体验，您可以**流式传输**代理的响应。代理会在生成文本块时逐步输出，而不是等待完整回复。这在需要实时显示输出的聊天界面中尤为有用。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## 概要

在本课中，您学习了如何：

- **创建一个提供程序**，通过 `AzureAIProjectAgentProvider` 连接到 Azure AI Foundry Agent 服务。
- 使用 `@tool` 装饰器 **定义一个工具**，以便代理可以调用您的 Python 函数。
- **运行代理**，使用用户消息并打印其响应。
- **流式传输响应**，实现实时输出。

在下一课中，我们将更深入地探索代理框架，并学习如何为代理提供更强大的工具和多步骤推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：  
本文档使用AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。虽然我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版本的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误释承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
